# Student 2 — Document Analyst: Police PDF Report Extraction

**Assignment:** AI for Engineers — Multimodal Crime / Incident Report Analyzer  
**Role:** Student 2 — Document Analyst  
**Pipeline:** Extract PDF text (pdfplumber → PyMuPDF → pytesseract OCR) → spaCy NER → Export CSV

### Required Output Columns
`Report_ID | Department | Doc_Type | Date | Program | Key_Detail`

---

## Dataset
**Arkansas Police Department 1033 Training Plan Proposals** — FOIA-released official police PDF from MuckRock.  
File: `LESO2.pdf` (75 pages, mix of text-based and scanned pages)

In [ ]:
# ============================================================
# CELL 1 — Install all dependencies
# ============================================================
!pip install pdfplumber pymupdf pytesseract pillow pdf2image spacy pandas -q
!python -m spacy download en_core_web_sm -q
print('All packages installed.')

In [ ]:
# ============================================================
# CELL 2 — Extract text from PDF
# Strategy:
#   1. Try pdfplumber (recommended for text-based PDFs)
#   2. Fall back to PyMuPDF (fitz) if pdfplumber returns nothing
#   3. Fall back to pytesseract OCR for scanned pages
# ============================================================
import pdfplumber
import fitz  # PyMuPDF
import os

PDF_PATH = 'LESO2.pdf'

def extract_with_pdfplumber(pdf_path):
    """Primary extraction: pdfplumber (best for text-based PDFs and tables)."""
    pages = {}
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text and text.strip():
                pages[i + 1] = text.strip()
    return pages

def extract_with_pymupdf(pdf_path):
    """Fallback extraction: PyMuPDF (handles more PDF variants)."""
    pages = {}
    doc = fitz.open(pdf_path)
    for i in range(len(doc)):
        text = doc[i].get_text('text').strip()
        if text:
            pages[i + 1] = text
    doc.close()
    return pages

def extract_with_ocr(pdf_path, page_numbers):
    """OCR fallback: pytesseract for scanned pages with no text layer."""
    try:
        import pytesseract
        from pdf2image import convert_from_path
        pages = {}
        images = convert_from_path(pdf_path, dpi=200)
        for pg_num in page_numbers:
            if pg_num <= len(images):
                text = pytesseract.image_to_string(images[pg_num - 1])
                if text.strip():
                    pages[pg_num] = text.strip()
        return pages
    except Exception as e:
        print(f'OCR not available: {e}')
        return {}

# --- Step 1: Try pdfplumber ---
print('Trying pdfplumber...')
extracted_pages = extract_with_pdfplumber(PDF_PATH)
print(f'  pdfplumber found text on {len(extracted_pages)} pages')

# --- Step 2: Fall back to PyMuPDF if pdfplumber found nothing ---
if not extracted_pages:
    print('Falling back to PyMuPDF...')
    extracted_pages = extract_with_pymupdf(PDF_PATH)
    print(f'  PyMuPDF found text on {len(extracted_pages)} pages')

# --- Step 3: Try OCR on remaining scanned pages ---
doc = fitz.open(PDF_PATH)
all_pages = set(range(1, len(doc) + 1))
doc.close()
scanned_pages = sorted(all_pages - set(extracted_pages.keys()))
print(f'  {len(scanned_pages)} scanned pages detected — attempting OCR...')
ocr_pages = extract_with_ocr(PDF_PATH, scanned_pages[:10])  # OCR first 10 scanned pages
extracted_pages.update(ocr_pages)
print(f'  OCR extracted text from {len(ocr_pages)} additional pages')

print(f'\nTotal pages with text: {len(extracted_pages)}')
for pg, text in sorted(extracted_pages.items()):
    print(f'  Page {pg}: {text[:80].replace(chr(10)," ")}...')

In [ ]:
# ============================================================
# CELL 3 — spaCy NER: extract dates, organizations, persons
# ============================================================
import spacy
import re

nlp = spacy.load('en_core_web_sm')

def extract_entities(text):
    """Extract named entities using spaCy NER."""
    doc = nlp(text[:2000])  # limit to avoid memory issues
    entities = {
        'orgs':    [e.text for e in doc.ents if e.label_ == 'ORG'],
        'persons': [e.text for e in doc.ents if e.label_ == 'PERSON'],
        'dates':   [e.text for e in doc.ents if e.label_ == 'DATE'],
        'locs':    [e.text for e in doc.ents if e.label_ in ('GPE', 'LOC', 'FAC')],
    }
    return entities

def normalize_date(raw_date):
    """Convert various date strings to YYYY-MM-DD."""
    patterns = [
        (r'(\d{1,2})/(\d{1,2})/(\d{2,4})', lambda m: f"{int(m.group(3)) if len(m.group(3))==4 else 2000+int(m.group(3))}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"),
        (r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{1,2}),?\s+(\d{4})',
         lambda m: f"{m.group(3)}-{['January','February','March','April','May','June','July','August','September','October','November','December'].index(m.group(1))+1:02d}-{int(m.group(2)):02d}"),
        (r'(\d{1,2})\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})',
         lambda m: f"{m.group(3)}-{['January','February','March','April','May','June','July','August','September','October','November','December'].index(m.group(2))+1:02d}-{int(m.group(1)):02d}"),
        (r'(\d{1,2})-(\d{1,2})-(\d{4})',
         lambda m: f"{m.group(3)}-{int(m.group(1)):02d}-{int(m.group(2)):02d}"),
    ]
    for pattern, formatter in patterns:
        match = re.search(pattern, raw_date, re.IGNORECASE)
        if match:
            try:
                return formatter(match)
            except Exception:
                continue
    return raw_date  # return as-is if no pattern matched

# Run NER on all extracted pages
page_entities = {}
for pg, text in sorted(extracted_pages.items()):
    ents = extract_entities(text)
    page_entities[pg] = ents
    print(f'Page {pg}: orgs={ents["orgs"][:2]} | persons={ents["persons"][:2]} | dates={ents["dates"][:2]}')

In [ ]:
# ============================================================
# CELL 4 — Segment PDF into logical reports
# Each report = a block of pages belonging to one department
# ============================================================
import re

# Define report segments based on document structure
# (identified by 'From:' or department header on cover page)
report_segments = [
    {'report_id': 'RPT_001', 'pages': [24, 25, 26]},
    {'report_id': 'RPT_002', 'pages': [27, 28, 29, 30]},
    {'report_id': 'RPT_003', 'pages': [32, 33]},
    {'report_id': 'RPT_004', 'pages': [34, 35]},
    {'report_id': 'RPT_005', 'pages': [66, 67, 68]},
]

def get_combined_text(pages_list):
    return ' '.join(extracted_pages.get(p, '') for p in pages_list)

def extract_field(text, field):
    """Extract 'From:' / 'Date:' / 'Ref:' fields from cover page text."""
    match = re.search(rf'{field}\s*:?\s*(.+)', text, re.IGNORECASE)
    return match.group(1).strip() if match else ''

def classify_doc_type(text):
    """Rule-based document type classification."""
    text_l = text.lower()
    if 'standard operating procedure' in text_l or ' sop' in text_l:
        return 'Standard Operating Procedure (SOP)'
    if 'course' in text_l or 'lesson' in text_l or 'training' in text_l:
        return '1033 MRAP Training Proposal'
    return 'Official Police Document'

def extract_program(text):
    """Identify the law enforcement program referenced."""
    text_l = text.lower()
    programs = []
    if 'leso 1033' in text_l or '1033 program' in text_l:
        programs.append('LESO 1033')
    if 'cleet' in text_l:
        programs.append('CLEET')
    if 'clest' in text_l:
        programs.append('CLEST')
    if 'swat' in text_l or 's.w.a.t' in text_l:
        programs.append('SWAT')
    return ' / '.join(programs) if programs else 'Law Enforcement Support'

def build_key_detail(text, doc_type):
    """Build a concise summary of the key detail."""
    detail_parts = []
    # Extract duration
    dur = re.search(r'(\d+\.?\d*)\s*hours?', text, re.IGNORECASE)
    if dur:
        detail_parts.append(f"{dur.group(1)}-hour training")
    # Extract trainees count
    trainees = re.search(r'number of trainees[:\s]+(\d+[\s\-]*\d*)', text, re.IGNORECASE)
    if trainees:
        detail_parts.append(f"{trainees.group(1).strip()} trainees")
    # Extract vehicle type
    if 'mrap' in text.lower():
        detail_parts.append('MRAP Vehicle Operations')
    if 'sop' in doc_type.lower():
        detail_parts.append('High-risk operations, natural disaster response')
    return '; '.join(detail_parts) if detail_parts else 'MRAP training documentation'

print('Segments identified:', [s['report_id'] for s in report_segments])

In [ ]:
# ============================================================
# CELL 5 — Build structured DataFrame
# ============================================================
import pandas as pd

rows = []

for seg in report_segments:
    text = get_combined_text(seg['pages'])
    ents = extract_entities(text)

    # Extract core fields
    raw_dept   = extract_field(text, 'From')
    raw_date   = extract_field(text, 'Date')
    doc_type   = classify_doc_type(text)
    program    = extract_program(text)
    key_detail = build_key_detail(text, doc_type)

    # Normalize date
    norm_date = normalize_date(raw_date) if raw_date else ents['dates'][0] if ents['dates'] else 'N/A'

    # Department: prefer 'From:' field, fall back to first ORG entity
    department = raw_dept if raw_dept else (ents['orgs'][0] if ents['orgs'] else 'Unknown Department')

    rows.append({
        'Report_ID':  seg['report_id'],
        'Department': department,
        'Doc_Type':   doc_type,
        'Date':       norm_date,
        'Program':    program,
        'Key_Detail': key_detail,
    })

pdf_df = pd.DataFrame(rows)
print('Extracted DataFrame:')
print(pdf_df.to_string(index=False))

In [ ]:
# ============================================================
# CELL 6 — Validate and save pdf_output.csv
# ============================================================

REQUIRED_COLUMNS = ['Report_ID', 'Department', 'Doc_Type', 'Date', 'Program', 'Key_Detail']

# Ensure correct column order
pdf_df = pdf_df[REQUIRED_COLUMNS]

# Validate
assert list(pdf_df.columns) == REQUIRED_COLUMNS, f'Column mismatch: {list(pdf_df.columns)}'
assert len(pdf_df) > 0, 'DataFrame is empty!'

pdf_df.to_csv('pdf_output.csv', index=False)

print('Saved pdf_output.csv')
print(f'  Rows: {len(pdf_df)} | Columns: {REQUIRED_COLUMNS}')
print()
print(pdf_df.to_string(index=False))